## Web Scraping

In [0]:
from datetime import datetime, timedelta
import requests

GNEWS_API_KEY = "your API key"
GNEWS_BASE_URL = "https://gnews.io/api/v4/search"

def query_gnews_with_fallback(
    company: str,
    company_aliases: list[str] = None,
    max_articles: int = 10,
    days_back: int = 90,
    min_results_threshold: int = 3,
    lang: str = "en",
    country: str = "us"
) -> list[dict]:
    """
    Tries a focused M&A query first. Falls back to a broader query
    if results are below threshold.
    """

    aliases = company_aliases or [company]
    name_clause = " OR ".join([f'"{a}"' for a in aliases])

    ma_terms = '(acquisition OR merger OR takeover OR buyout OR "strategic deal" OR investor OR "deal close")'
    
    queries = [
        f"({name_clause}) AND {ma_terms}",  # tight — M&A focused
        f"({name_clause})",                  # broad — any coverage
    ]

    for i, query in enumerate(queries):
        params = {
            "q":       query,
            "lang":    lang,
            "country": country,
            "max":     max_articles,
            "from":    (datetime.utcnow() - timedelta(days=days_back)).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "apikey":  GNEWS_API_KEY,
            "expand":  "content",
        }

        try:
            response = requests.get(GNEWS_BASE_URL, params=params, timeout=10)
            response.raise_for_status()
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code
            if status == 403:
                raise RuntimeError("GNews: daily request limit reached or invalid API key.")
            elif status == 429:
                raise RuntimeError("GNews: rate limit hit — add sleep between calls.")
            else:
                raise RuntimeError(f"GNews HTTP error {status} for '{company}'")
        except requests.exceptions.Timeout:
            raise RuntimeError(f"GNews: request timed out for '{company}'")

        data = response.json()
        if "errors" in data:
            raise RuntimeError(f"GNews API error: {data['errors']}")

        articles = data.get("articles", [])

        if len(articles) >= min_results_threshold or i == len(queries) - 1:
            query_type = "M&A" if i == 0 else "broad fallback"
            print(f"{company}: {len(articles)} articles ({query_type})")
            return [
                {
                    "title":       a.get("title", "").strip(),
                    "date":        a.get("publishedAt", ""),
                    "url":         a.get("url", ""),
                    "source":      a.get("source", {}).get("name", "Unknown"),
                    "snippet":     a.get("description", "").strip(),
                    "body":        a.get("content", "").strip(),
                    "company_tag": company,
                    "query_type":  query_type,
                }
                for a in articles
            ]

    return []

In [0]:
company_configs = [
    ("Norfolk Southern",  ["Norfolk Southern", "NSC railroad"],        "us"),
    ("Warner Bros.",      ["Warner Bros", "Warner Brothers", "WBD"],   "us"),
    ("Kenvue",            ["Kenvue"],                                  "us"),
    ("Schroders",         ["Schroders", "Schroders plc"],              "us"),
    ("Arcellx",           ["Arcellx", "ACLX"],                   "us"),
    ("Coterra Energy", ["Coterra Energy", "CTRA"],                     "us"),
    ("InPost",    ["InPost", "InPost S.A.", "INPST"],                  None),
    ("Webster Financial", ["Webster Financial", "Webster Bank"],       "us"),
    ("Urbaser",           ["Urbaser", "Urbaser waste management"],     None),
    ("Comerica",          ["Comerica"],                                "us"),
]

all_results = {}
for company, aliases, country in company_configs:
    results = query_gnews_with_fallback(
        company,
        company_aliases=aliases,
        max_articles=10,
        days_back=90,
        country=country
    )
    all_results[company] = results

/root/.ipykernel/20849/command-8240509881953031-938204905:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "from":    (datetime.utcnow() - timedelta(days=days_back)).strftime("%Y-%m-%dT%H:%M:%SZ"),


Norfolk Southern: 10 articles (M&A)
Warner Bros.: 10 articles (M&A)
Kenvue: 10 articles (broad fallback)
Schroders: 7 articles (M&A)
Arcellx: 10 articles (broad fallback)
Coterra Energy: 8 articles (broad fallback)
InPost: 10 articles (M&A)
Webster Financial: 6 articles (M&A)
Urbaser: 6 articles (broad fallback)
Comerica: 7 articles (M&A)


In [0]:
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import urlparse
from difflib import SequenceMatcher

# Sites known to be paywalled or bot-protected — scraping these is futile
BLOCKED_DOMAINS = {
    "wsj.com", "ft.com", "bloomberg.com", "barrons.com",
    "thetimes.co.uk", "telegraph.co.uk", "economist.com",
    "hbr.org", "seekingalpha.com"
}

# Rotating user agents to reduce bot detection
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
]

def is_blocked_domain(url: str) -> bool:
    """Check if URL belongs to a known paywalled/blocked domain."""
    domain = urlparse(url).netloc.lower().replace("www.", "")
    return any(domain.endswith(blocked) for blocked in BLOCKED_DOMAINS)

def is_paywall_response(text: str) -> bool:
    """Heuristic check for paywalled or gated content."""
    paywall_signals = [
        "subscribe to read",
        "subscription required",
        "sign in to read",
        "create a free account",
        "already a subscriber",
        "to continue reading",
        "unlock this article",
    ]
    sample = text.lower()[:2000]  # only check top of page
    return any(signal in sample for signal in paywall_signals)

def is_content_sufficient(text: str, min_chars: int = 300) -> bool:
    """Check if scraped body is long enough to be useful."""
    return len(text.strip()) >= min_chars

def extract_body(html: str) -> str:
    """
    Extract clean article body from raw HTML.
    Targets semantic article tags first, falls back to
    paragraph aggregation with boilerplate filtering.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Remove noise elements
    for tag in soup(["script", "style", "nav", "header", "footer",
                      "aside", "figure", "figcaption", "form", "iframe"]):
        tag.decompose()

    # Try semantic containers first (most modern news sites use these)
    for selector in ["article", "main", '[role="main"]',
                     ".article-body", ".story-body", ".post-content",
                     ".entry-content", ".article-content"]:
        container = soup.select_one(selector)
        if container:
            paragraphs = container.find_all("p")
            text = " ".join(p.get_text(separator=" ", strip=True) for p in paragraphs)
            if is_content_sufficient(text):
                return text

    # Fallback — aggregate all <p> tags, filter short/boilerplate ones
    all_paragraphs = soup.find_all("p")
    filtered = [
        p.get_text(separator=" ", strip=True)
        for p in all_paragraphs
        if len(p.get_text(strip=True)) > 40  # skip nav snippets, captions
    ]
    return " ".join(filtered)

def scrape_article(url: str, retries: int = 2) -> dict:
    """
    Scrape a single article URL. Returns a dict with:
        body, scrape_status, char_count
    
    scrape_status values:
        "success"          — clean body extracted
        "paywall"          — content gated
        "blocked_domain"   — known paywall site, not attempted
        "insufficient"     — scraped but too short to be useful
        "error"            — request or parse failure
    """
    if is_blocked_domain(url):
        return {"body": "", "scrape_status": "blocked_domain", "char_count": 0}

    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
    }

    for attempt in range(retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code == 403:
                return {"body": "", "scrape_status": "blocked_domain", "char_count": 0}
            if response.status_code == 429:
                wait = 5 * (attempt + 1)
                print(f"  Rate limited on {url} — waiting {wait}s")
                time.sleep(wait)
                continue

            response.raise_for_status()

            if is_paywall_response(response.text):
                return {"body": "", "scrape_status": "paywall", "char_count": 0}

            body = extract_body(response.text)

            if not is_content_sufficient(body):
                return {"body": body, "scrape_status": "insufficient", "char_count": len(body)}

            return {"body": body, "scrape_status": "success", "char_count": len(body)}

        except requests.exceptions.Timeout:
            if attempt == retries:
                return {"body": "", "scrape_status": "error", "char_count": 0}
            time.sleep(2)

        except Exception as e:
            return {"body": "", "scrape_status": "error", "char_count": 0}

    return {"body": "", "scrape_status": "error", "char_count": 0}


def is_similar_title(title1: str, title2: str, threshold: float = 0.85) -> bool:
    """Catch syndicated duplicates with slightly different URLs but same title."""
    return SequenceMatcher(None, title1.lower(), title2.lower()).ratio() > threshold

def deduplicate_articles(articles: list[dict]) -> list[dict]:
    """Deduplicate by URL first, then by title similarity."""
    seen_urls = set()
    seen_titles = []
    unique = []
    
    for article in articles:
        url = article.get("url", "")
        title = article.get("title", "")
        
        if url in seen_urls:
            continue
            
        if any(is_similar_title(title, seen) for seen in seen_titles):
            continue
        
        seen_urls.add(url)
        seen_titles.append(title)
        unique.append(article)
    
    return unique

def is_relevant_article(article: dict, company: str) -> bool:
    """
    Check if the article is actually about the target company,
    not just mentioning it in passing.
    Checks title and first 300 chars of body/snippet.
    """
    company_core = company.lower().split()[0]  # "Norfolk" from "Norfolk Southern"
    
    title = article.get("title", "").lower()
    snippet = article.get("snippet", "").lower()[:300]
    body = article.get("body", "").lower()[:300]
    
    text = f"{title} {snippet} {body}"
    
    # Count how many times the company name appears
    mentions = text.count(company_core)
    
    # Title mention is a strong signal
    title_mention = company_core in title
    
    return title_mention or mentions >= 2

def scrape_all_articles(all_results: dict, delay_range: tuple = (1.5, 3.5)) -> list[dict]:
    records = []

    for company, articles in all_results.items():

        # Step 1 — deduplicate
        unique_articles = deduplicate_articles(articles)
        
        # Step 2 — relevance filter
        relevant_articles = [a for a in unique_articles if is_relevant_article(a, company)]
        
        dupes = len(articles) - len(unique_articles)
        irrelevant = len(unique_articles) - len(relevant_articles)
        print(f"\n--- {company} ({len(relevant_articles)} articles, {dupes} dupes, {irrelevant} irrelevant removed) ---")

        for article in relevant_articles:
            url = article.get("url", "")
            gnews_body = article.get("body", "")
            gnews_snippet = article.get("snippet", "")

            time.sleep(random.uniform(*delay_range))

            scrape_result = scrape_article(url)
            status = scrape_result["scrape_status"]
            print(f"  [{status}] {article['title'][:60]}...")

            # Body resolution: scraped > gnews body > snippet > empty
            if status == "success":
                final_body = scrape_result["body"]
            elif status in ("blocked_domain", "paywall", "insufficient", "error"):
                if gnews_body and is_content_sufficient(gnews_body):
                    final_body = gnews_body
                    status = "gnews_body_fallback"
                elif gnews_snippet:
                    final_body = gnews_snippet
                    status = "snippet_fallback"
                else:
                    final_body = ""
                    status = "no_content"
            else:
                final_body = ""
                status = "no_content"

            records.append({
                "company_tag":   company,
                "title":         article["title"],
                "date":          article["date"],
                "source":        article["source"],
                "url":           url,
                "body":          final_body,
                "char_count":    len(final_body),
                "scrape_status": status,
                "query_type":    article.get("query_type", "unknown"),
            })

    return records


# --- Run it ---
records = scrape_all_articles(all_results)

# Quick summary
from collections import Counter
status_counts = Counter(r["scrape_status"] for r in records)
print("\n--- Scrape Summary ---")
for status, count in status_counts.most_common():
    print(f"  {status}: {count}")


--- Norfolk Southern (2 articles, 8 dupes, 0 irrelevant removed) ---
  [success] Norfolk Southern, Union Pacific try railroad merger bid agai...
  [success] Union Pacific argues for its $85B acquisition of Norfolk Sou...

--- Warner Bros. (7 articles, 1 dupes, 2 irrelevant removed) ---
  [success] Consumers sue to block Paramount-Warner Bros. deal...
  [success] Paramount Sued by Its Own Subscribers...
  [success] What Stands in the Way of the Paramount-WBD Merger?...
  [success] Gunnar Wiedenfels Warner Bros. Finance Chief Through April 2...
  [success] Warner Bros Discovery David Zaslav Salary and Compensation f...
  [success] Paramount wants FCC to approve increased foreign ownership i...
  [success] Paramount wants FCC to approve higher foreign ownership in W...

--- Kenvue (10 articles, 0 dupes, 0 irrelevant removed) ---
  [success] Kenvue Stock: Is Wall Street Bullish or Bearish?...
  [success] Here's What to Expect From Kenvue's Next Earnings Report...
  [success] RBC Capital S

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from pyspark.sql.functions import col, to_timestamp, current_timestamp
import pandas as pd

# --- Schema ---
schema = StructType([
    StructField("company_tag",   StringType(),   nullable=False),
    StructField("title",         StringType(),   nullable=True),
    StructField("date",          StringType(),   nullable=True),
    StructField("source",        StringType(),   nullable=True),
    StructField("url",           StringType(),   nullable=True),
    StructField("body",          StringType(),   nullable=True),
    StructField("char_count",    IntegerType(),  nullable=True),
    StructField("scrape_status", StringType(),   nullable=True),
    StructField("query_type",    StringType(),   nullable=True),
])

# --- Convert records to Spark DataFrame ---
pdf = pd.DataFrame(records)

# Ensure char_count is int, not numpy int64 which Spark can mishandle
pdf["char_count"] = pdf["char_count"].astype(int)

# Drop any records with no usable body
pdf = pdf[pdf["body"].str.strip().str.len() > 0].reset_index(drop=True)

sdf = spark.createDataFrame(pdf, schema=schema)

# --- Parse and enrich ---
sdf = (
    sdf
    .withColumn("published_at", to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("ingested_at",  current_timestamp())
    .drop("date")
)

# --- Write to Delta ---
TABLE_NAME = "isa632_7474656346303369.plattdg.news_articles"

(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("company_tag")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Written {sdf.count()} records to {TABLE_NAME}")

# --- Validation queries ---
spark.sql(f"""
    SELECT 
        company_tag,
        COUNT(*)                        AS article_count,
        AVG(char_count)                 AS avg_char_count,
        COUNT(CASE WHEN body IS NULL 
              OR body = '' 
              THEN 1 END)               AS empty_bodies,
        MIN(published_at)               AS earliest_article,
        MAX(published_at)               AS latest_article
    FROM {TABLE_NAME}
    GROUP BY company_tag
    ORDER BY article_count DESC
""").show(truncate=False)

Written 64 records to isa632_7474656346303369.plattdg.news_articles
+-----------------+-------------+------------------+------------+-------------------+-------------------+
|company_tag      |article_count|avg_char_count    |empty_bodies|earliest_article   |latest_article     |
+-----------------+-------------+------------------+------------+-------------------+-------------------+
|Kenvue           |10           |3813.2            |0           |2026-02-17 21:45:00|2026-04-28 10:07:41|
|Arcellx          |10           |1582.5            |0           |2026-02-23 12:51:28|2026-04-17 13:17:25|
|Coterra Energy   |8            |1719.5            |0           |2026-02-03 12:37:15|2026-05-01 02:59:36|
|Warner Bros.     |7            |6116.714285714285 |0           |2026-04-28 19:57:15|2026-05-01 19:26:21|
|Schroders        |7            |3003.1428571428573|0           |2026-02-12 09:37:56|2026-02-18 06:30:01|
|Webster Financial|6            |4165.0            |0           |2026-02-03 18:38:00

In [0]:
# Read back from Delta and export to CSV
TABLE_NAME = "isa632_7474656346303369.plattdg.news_articles"

display(spark.read.format('delta').table(TABLE_NAME))

company_tag title source url body char_count scrape_status query_type published_at ingested_at Warner Bros. Consumers sue to block Paramount-Warner Bros. deal Los Angeles Times https://www.latimes.com/entertainment-arts/business/story/2026-05-01/consumers-sue-to-block-paramount-warner-bros-deal This is read by an automated voice. Please report any issues or inconsistencies here . A group of five consumers have filed a lawsuit against Paramount Skydance seeking to block its acquisition of Warner Bros. Discovery and unwind the earlier merger that joined the storied Melrose Avenue studio with David Ellison’s Skydance Media, alleging that both deals reduce marketplace competition. The lawsuit, filed Thursday in U.S. District Court in the Northern District of California, alleges the Paramount-Warner deal will lead to increased prices, fewer consumer choices and reduce production of film and TV since a major rival in the entertainment business will be eliminated. The suit also alleges that the Paramount-Skydance merger, which was finalized last year , led to higher prices for the Paramount+ streaming service. The plaintiffs — Pamela Faust, Len Marazzo, Lisa McCarthy, Deborah Rubinsohn and Gary Talewsky — are either Paramount+ subscribers, pay for cable bundles that include Paramount-owned TV channels or are moviegoers who watch films in theaters. The deal activity for Paramount is part of a growing list of recent media mergers, including Walt Disney Co.’s 2019 acquisition of much of 21st Century Fox and Amazon’s purchase of MGM in 2021 . “These acquisitions show an industry moving by successive combinations toward fewer independent rivals, exactly the consolidation backdrop that heightens the competitive threat posed by the next merger, even if the combined firm remains smaller than the largest platforms,” the lawsuit states. Paramount is aware of the lawsuit and “confident that it is without merit,” a company spokesperson said. “The combination of Paramount and [Warner Bros. Discovery] will create a stronger competitor that is well positioned to serve as a champion for creative talent and consumer choice,” the spokesperson said in a statement. The Paramount-Warner deal is currently winding its way through regulatory approvals. While that process is underway, Paramount has asked the Federal Communications Commission for permission to exceed a cap on foreign ownership for U.S. media companies. Paramount expects to receive $24 billion in funds from three Middle Eastern royal families, who will become part owners of the combined company. Those total funds will represent about 49% of equity in that new company, exceeding the current foreign ownership cap of 25%. Paramount has said the Ellison family and RedBird Capital Partners “collectively hold the largest equity stake in the combined company and continue to be the sole owners of Class A Common Stock, representing 100% of the voting shares.” But on Friday, Rep. Sam Liccardo (D- San Jose) urged the FCC to deny Paramount’s petition on the foreign ownership aspect of the deal. “Congress did not entrust the public airwaves to this agency so that it could auction off America to Riyadh, Abu Dhabi and Doha,” he wrote in a statement. “This will not stand.” 2953 success M&A 2026-05-01T19:26:21Z 2026-05-03T21:58:04.580869Z Warner Bros. Paramount Sued by Its Own Subscribers Vulture https://www.vulture.com/article/paramount-sued-by-its-own-subscribers.html The shareholders might be loving the idea of Paramount’s bid on Warner Bros. Discovery, but subscribers are not. On April 30, Paramount subscribers filed a lawsuit in California federal court against the studio in an attempt to both block the merger and undo aspects of Skydance’s acquisition from 2024. The suit represents three current Paramount+ subscribers and two “prospective” subscribers (presumably waiting to see how Survivor 50 wraps up before biting the bullet), all of whom feel that this merger, when taken in context of the Amazon-MGM d